# ⚽ YOLOv11 Fine-Tuning for Football Video Analytics
### High-Precision Multi-Class Object Detection (Players, Referees, Ball & Goalkeepers)

This Google Colab notebook provides the end-to-end training, validation, and export pipeline for fine-tuning **Ultralytics YOLOv11** on broadcast/drone football pitch imagery.

#### Key Highlights:
- **Base Architecture**: YOLOv11 (`yolo11n.pt` or `yolo11s.pt`)
- **Target Classes**: `player` (0), `referee` (1), `ball` (2), `goalkeeper` (3)
- **Optimizer**: AdamW with Cosine Annealing learning rate schedule
- **Domain Augmentations**: Mosaic (0.8), HSV color jitter (to handle varied lighting/grass shades), Horizontal Flip
- **Export Target**: `.pt` and ONNX for direct plug-and-play into `backend/weights/`

In [ ]:
# Step 1: Verify Hardware Accelerator (GPU)
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

In [ ]:
# Step 2: Install Ultralytics and Supporting Vision Dependencies
!pip install -q --upgrade ultralytics supervision roboflow opencv-python pyyaml matplotlib

## 📂 Dataset Acquisition and Preparation
You can load a public football dataset (such as Football Players Detection from Roboflow Universe) or provide your own annotated dataset in YOLO format.

Expected directory format:
```
dataset/
  ├── train/
  │   ├── images/
  │   └── labels/
  ├── valid/
  │   ├── images/
  │   └── labels/
  └── data.yaml
```

In [ ]:
# Step 3: Dataset Configuration / Download
import os
import yaml

# OPTION A: Download via Roboflow (Uncomment and insert your API key)
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
# version = project.version(1)
# dataset = version.download("yolov11")
# data_yaml_path = f"{dataset.location}/data.yaml"

# OPTION B: Write a custom or local football data.yaml
os.makedirs("football_dataset/train/images", exist_ok=True)
os.makedirs("football_dataset/train/labels", exist_ok=True)
os.makedirs("football_dataset/valid/images", exist_ok=True)
os.makedirs("football_dataset/valid/labels", exist_ok=True)

data_config = {
    "path": os.path.abspath("football_dataset"),
    "train": "train/images",
    "val": "valid/images",
    "names": {
        0: "player",
        1: "referee",
        2: "ball",
        3: "goalkeeper"
    }
}

data_yaml_path = "football_dataset/data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Data YAML created at {data_yaml_path}:")
with open(data_yaml_path) as f:
    print(f.read())

## 🚀 Training YOLOv11 with Domain-Specific Hyperparameters
Football pitches exhibit large aspect ratios, rapid camera pans, and small ball dimensions ($< 15\times 15$ pixels). We employ:
- **Base Model**: `yolo11n.pt` for high frame rates (or `yolo11s.pt` for higher AP)
- **Image Size**: `640` (standard) or `1280` (optimal for tiny ball localization)
- **Loss Weighting**: Enhanced box and class loss to prevent referee/player cross-confusion
- **AdamW Optimizer**: Better regularization than standard SGD

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11 Nano backbone
model = YOLO("yolo11n.pt")

# Initiate Fine-Tuning
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    mosaic=0.8,            # Effective for contextual pitch awareness
    mixup=0.1,             # Occlusion robustness
    hsv_h=0.015,           # Grass / jersey color preservation
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,            # Symmetric pitch play
    device=0 if torch.cuda.is_available() else "cpu",
    project="football_experiments",
    name="yolo11_finetuned",
    save=True,
    plots=True
)

## 📊 Model Evaluation and Validation Metrics
Evaluate the fine-tuned model against validation splits to measure class-wise Precision, Recall, and mAP.

In [ ]:
# Evaluate on validation split
metrics = model.val(data=data_yaml_path, split="val")

print("\n===== VALIDATION METRICS SUMMARY =====")
print(f"Overall mAP@50:    {metrics.box.map50:.4f}")
print(f"Overall mAP@50-95: {metrics.box.map:.4f}")
print(f"Mean Precision:    {metrics.box.mp:.4f}")
print(f"Mean Recall:       {metrics.box.mr:.4f}")

# Class-wise performance
for idx, class_name in enumerate(["player", "referee", "ball", "goalkeeper"]):
    if idx < len(metrics.box.maps):
        print(f"- {class_name:12s} | mAP50: {metrics.box.maps[idx]:.4f}")

## 🔍 Qualitative Test Inference and Visual Verification

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

# Pick a sample validation image
sample_images = glob.glob("football_dataset/valid/images/*.jpg") + glob.glob("football_dataset/valid/images/*.png")
if sample_images:
    test_img = sample_images[0]
    res = model.predict(source=test_img, conf=0.25, iou=0.45)
    
    # Plot detections
    annotated_frame = res[0].plot()
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Fine-Tuned YOLOv11 Detection Output")
    plt.show()
else:
    print("Add test frames to visualize predictions.")

## 💾 Checkpoint Export and Deployment Instructions

1. Download the exported weights (`best.pt` or `best.onnx`).
2. Transfer the file to your local repository directory: `backend/weights/yolo11_football_finetuned.pt`.
3. Configure your `backend/.env` file:
   ```env
   YOLO_MODEL_PATH=weights/yolo11_football_finetuned.pt
   ```
4. Restart the backend. The `PlayerDetector` will automatically detect the custom multi-class weights and map `player`, `referee`, `ball`, and `goalkeeper` seamlessly!

In [ ]:
import shutil

# Locate best checkpoint
best_weights = "football_experiments/yolo11_finetuned/weights/best.pt"
if os.path.exists(best_weights):
    # Export to ONNX
    exported_onnx = model.export(format="onnx", dynamic=True)
    print(f"Exported ONNX: {exported_onnx}")
    
    # Download weights directly in Google Colab
    try:
        from google.colab import files
        files.download(best_weights)
        print("Triggered download of best.pt")
    except ImportError:
        print(f"Weights saved at {os.path.abspath(best_weights)}")
else:
    print("Training weights not found at path.")